In [1]:
import polars as pl

order_df = pl.read_csv(r"/mnt/d/VeeraPraveen.S/learn/projects/Insights-from-Failed-Orders/datasets/data_orders.csv")
offer_df = pl.read_csv(r"/mnt/d/VeeraPraveen.S/learn/projects/Insights-from-Failed-Orders/datasets/data_offers.csv")

# Plot the distribution on the failure reasons
- cancellations before and after driver assignment, and reasons for order rejection. 
- Analyse the resulting plot. Which category has the highest number of orders?

In [2]:
order_df.head(10)

order_datetime,origin_longitude,origin_latitude,m_order_eta,order_gk,order_status_key,is_driver_assigned_key,cancellations_time_in_seconds
str,f64,f64,f64,i64,i64,i64,f64
"""18:08:07""",-0.978916,51.456173,60.0,3000583041974,4,1,198.0
"""20:57:32""",-0.950385,51.456843,null,3000583116437,4,0,128.0
"""12:07:50""",-0.96952,51.455544,477.0,3000582891479,4,1,46.0
"""13:50:20""",-1.054671,51.460544,658.0,3000582941169,4,1,62.0
"""21:24:45""",-0.967605,51.458236,null,3000583140877,9,0,null
"""21:21:23""",-0.947011,51.45638,null,3000583117054,9,0,null
"""07:58:15""",-0.955637,51.470372,null,3000582791789,9,0,null
"""07:53:46""",-0.97823,51.454575,null,3000582791562,9,0,null
"""08:53:01""",-1.052298,51.454308,null,3000582817606,9,0,null


In [3]:
distribution = order_df.with_columns(
    pl.when(pl.col("is_driver_assigned_key") == 1)
    .then(pl.lit("Driver Assigned"))
    .otherwise(pl.lit("Driver Not Assigned"))
    .alias("order_driver_status")
)

In [4]:
distribution = (
    distribution.group_by(pl.col("order_status_key"), "order_driver_status")
    .agg([pl.len().alias("No of cancelled orders")])
    .with_columns(
        pl.when(pl.col("order_status_key") == 4)
        .then(pl.lit("cancelled by client"))
        .otherwise(pl.lit("cancelled by system"))
        .alias("Reasons for Cancelled")
    )
)

In [5]:
distribution

order_status_key,order_driver_status,No of cancelled orders,Reasons for Cancelled
i64,str,u32,str
9,"""Driver Not Assigned""",3406,"""cancelled by system"""
9,"""Driver Assigned""",3,"""cancelled by system"""
4,"""Driver Assigned""",2811,"""cancelled by client"""
4,"""Driver Not Assigned""",4496,"""cancelled by client"""


## PIE Plots

In [ ]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots


analysis_df = distribution.group_by(["Reasons for Cancelled", "order_driver_status"]).agg(
    pl.col("No of cancelled orders").sum().alias("cancelled_orders")
)


# ============================================================
# CREATE FIGURE
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "domain"}, {"type": "bar"}]],
    column_widths=[0.30, 0.60],
    subplot_titles=["Cancellation Distribution", "Cancellation Analysis"],
)


# ============================================================
# PIE - CLIENT
# ============================================================

client_df = analysis_df.filter(pl.col("Reasons for Cancelled") == "cancelled by client")

fig.add_trace(
    go.Pie(
        labels=client_df["order_driver_status"].to_list(),
        values=client_df["cancelled_orders"].to_list(),
        textinfo="label+percent",
        sort=False,
        name="Client",
    ),
    row=1,
    col=1,
)


# ============================================================
# PIE - SYSTEM
# ============================================================

system_df = analysis_df.filter(pl.col("Reasons for Cancelled") == "cancelled by system")

fig.add_trace(
    go.Pie(
        labels=system_df["order_driver_status"].to_list(),
        values=system_df["cancelled_orders"].to_list(),
        textinfo="label+percent",
        sort=False,
        name="System",
        visible=False,
    ),
    row=1,
    col=1,
)


# ============================================================
# FIXED BAR CHART
# ============================================================

# Driver Not Assigned
not_assigned = analysis_df.filter(pl.col("order_driver_status") == "Driver Not Assigned")

fig.add_trace(
    go.Bar(
        x=not_assigned["Reasons for Cancelled"].to_list(),
        y=not_assigned["cancelled_orders"].to_list(),
        name="Driver Not Assigned",
        text=not_assigned["cancelled_orders"].to_list(),
        textposition="outside",
    ),
    row=1,
    col=2,
)


# Driver Assigned
assigned = analysis_df.filter(pl.col("order_driver_status") == "Driver Assigned")

fig.add_trace(
    go.Bar(
        x=assigned["Reasons for Cancelled"].to_list(),
        y=assigned["cancelled_orders"].to_list(),
        name="Driver Assigned",
        text=assigned["cancelled_orders"].to_list(),
        textposition="outside",
    ),
    row=1,
    col=2,
)


# ============================================================
# LAYOUT
# ============================================================

fig.update_layout(
    title="Cancelled Order Analysis",
    width=1400,
    height=600,
    autosize=False,
    margin=dict(
        l=80,
        r=50,
        t=120,
        b=50,
    ),
    # Keep the bar chart grouped
    barmode="group",
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            buttons=[
                dict(
                    label="Cancelled by Client",
                    method="update",
                    args=[
                        {
                            "visible": [
                                True,  # client pie
                                False,  # system pie
                                True,  # fixed bar - not assigned
                                True,  # fixed bar - assigned
                            ]
                        },
                        {"title": "Cancelled Order Analysis — Client"},
                    ],
                ),
                dict(
                    label="Cancelled by System",
                    method="update",
                    args=[
                        {
                            "visible": [
                                False,  # client pie
                                True,  # system pie
                                True,  # fixed bar - not assigned
                                True,  # fixed bar - assigned
                            ]
                        },
                        {"title": "Cancelled Order Analysis — System"},
                    ],
                ),
            ],
            x=0.5,
            y=1.12,
            xanchor="center",
            yanchor="top",
        )
    ],
)


fig.show()

### Based on the analysis the Driver not assigned cancellation occuring more in the system 
- cancelled by the system happend 99.99% - 3406
- cancelled by Client happend 61.5 % - 4496

# Plot the distribution of failed orders by hours

# 

In [ ]:
#